In [2]:
from datetime import datetime, timedelta
import random
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration for Big Data volume
NUM_CUSTOMERS = 10000
NUM_PRODUCTS = 500
NUM_ORDERS = 50000
NUM_EVENTS = 150000

# ---------------------------------------------------------
# 1. Indian City & Pincode Mapping per State
# ---------------------------------------------------------
state_city_pincode = {
    "Maharashtra": [
        ("Mumbai", "400001"),
        ("Pune", "411001"),
        ("Nagpur", "440001"),
        ("Nashik", "422001"),
    ],
    "Uttar Pradesh": [
        ("Lucknow", "226001"),
        ("Kanpur", "208001"),
        ("Varanasi", "221001"),
        ("Agra", "282001"),
    ],
    "Karnataka": [
        ("Bengaluru", "560001"),
        ("Mysuru", "570001"),
        ("Hubballi", "580020"),
    ],
    "Gujarat": [
        ("Ahmedabad", "380001"),
        ("Surat", "395001"),
        ("Vadodara", "390001"),
    ],
    "Tamil Nadu": [
        ("Chennai", "600001"),
        ("Coimbatore", "641001"),
        ("Madurai", "625001"),
    ],
    "West Bengal": [
        ("Kolkata", "700001"),
        ("Howrah", "711101"),
        ("Siliguri", "734001"),
    ],
    "Bihar": [
        ("Patna", "800001"),
        ("Gaya", "823001"),
        ("Muzaffarpur", "842001"),
    ],
    "Rajasthan": [
        ("Jaipur", "302001"),
        ("Jodhpur", "342001"),
        ("Udaipur", "313001"),
    ],
    "Madhya Pradesh": [
        ("Bhopal", "462001"),
        ("Indore", "452001"),
        ("Gwalior", "474001"),
    ],
    "Telangana": [
        ("Hyderabad", "500001"),
        ("Warangal", "506001"),
        ("Nizamabad", "503001"),
    ],
}

channels = [
    "Organic Search",
    "Paid Ads (Instagram/FB)",
    "Referral / WhatsApp Share",
    "Direct App Open",
    "Affiliate",
]
first_names = [
    "Aarav",
    "Priya",
    "Rohan",
    "Ananya",
    "Vikram",
    "Sneha",
    "Kabir",
    "Neha",
    "Arjun",
    "Pooja",
    "Rahul",
    "Shreya",
    "Siddharth",
    "Kavya",
    "Aditya",
]
last_names = [
    "Sharma",
    "Patel",
    "Mehta",
    "Iyer",
    "Singh",
    "Kulkarni",
    "Joshi",
    "Verma",
    "Nair",
    "Deshmukh",
    "Gupta",
    "Rao",
]

# ---------------------------------------------------------
# 2. Customers Table
# ---------------------------------------------------------
customer_data = []
start_date = datetime(2024, 1, 1)

for i in range(1, NUM_CUSTOMERS + 1):
    c_id = f"CUST-{i:05d}"
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    state = random.choice(list(state_city_pincode.keys()))
    city, pincode = random.choice(state_city_pincode[state])
    channel = random.choice(channels)
    signup = start_date + timedelta(days=random.randint(0, 600))

    customer_data.append(
        [
            c_id,
            name,
            state,
            city,
            pincode,
            channel,
            signup.strftime("%Y-%m-%d"),
        ]
    )

df_customers = pd.DataFrame(
    customer_data,
    columns=[
        "Customer_ID",
        "Customer_Name",
        "Customer_State",
        "City",
        "Pincode",
        "Acquisition_Channel",
        "Signup_Date",
    ],
)

# ---------------------------------------------------------
# 3. Products & Sellers Table
# ---------------------------------------------------------
sellers = [f"SELLER-{s:04d}" for s in range(101, 201)]  # 100 Meesho Sellers

categories = {
    "Ethnic Wear": ["Kurti", "Saree", "Lehenga", "Anarkali Suits"],
    "Western Wear": ["Jeans", "Tops", "Dresses", "Jackets"],
    "Home & Kitchen": ["Bedsheets", "Curtains", "Cookware", "Kitchen Tools"],
    "Beauty & Footwear": ["Lipstick", "Skincare Combo", "Flats", "Sneakers"],
    "Jewelry & Accessories": [
        "Earrings",
        "Necklace Set",
        "Handbags",
        "Smartwatches",
    ],
}

product_data = []
p_counter = 101

for cat, subcats in categories.items():
    for subcat in subcats:
        for _ in range(25):  # 25 products per subcategory = 500 products
            sku = f"SKU-{p_counter}"
            p_name = f"{subcat} - Model {random.randint(10, 99)}"
            seller_id = random.choice(sellers)
            supplier_price = round(
                random.uniform(80, 450), 2
            )  # Cost to supplier
            base_price = round(
                supplier_price * random.uniform(1.3, 2.2), 2
            )  # Product MRP/Base Price
            discount_pct = (
                round(random.uniform(5.0, 30.0), 1)
                if random.random() > 0.3
                else 0.0
            )
            rating = round(random.uniform(3.2, 4.9), 1)
            stock = random.randint(50, 1000)

            product_data.append([
                sku,
                p_name,
                cat,
                subcat,
                seller_id,
                supplier_price,
                base_price,
                discount_pct,
                rating,
                stock,
            ])
            p_counter += 1

df_products = pd.DataFrame(
    product_data,
    columns=[
        "SKU",
        "Product_Name",
        "Product_Category",
        "Subcategory",
        "Seller_ID",
        "Supplier_Price",
        "Base_Price",
        "Discount_%",
        "Product_Rating",
        "Stock_Qty",
    ],
)

# ---------------------------------------------------------
# 4. Orders & Order Line Items Tables
# ---------------------------------------------------------
payment_methods = [
    "UPI (GPay/PhonePe)",
    "Cash on Delivery (COD)",
    "Debit Card",
    "Credit Card",
    "Net Banking",
]
order_statuses = [
    "DELIVERED",
    "DELIVERED",
    "DELIVERED",
    "DELIVERED",
    "RTO_COMPLETE",
    "CANCELLED",
    "SHIPPED",
]

order_data = []
order_item_data = []

order_start = datetime(2025, 1, 1)

for o_idx in range(1, NUM_ORDERS + 1):
    o_id = f"ORD-{o_idx:06d}"

    # Sample random customer to retrieve location metadata
    cust = df_customers.sample(1).iloc[0]
    c_id = cust["Customer_ID"]
    state = cust["Customer_State"]
    city = cust["City"]
    pincode = cust["Pincode"]

    o_date = order_start + timedelta(
        days=random.randint(0, 365),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
    )
    pay_method = random.choice(payment_methods)
    status = random.choice(order_statuses)

    # 1 to 3 items per order
    num_items = random.choices([1, 2, 3], weights=[0.7, 0.2, 0.1])[0]
    total_order_amt = 0

    for _ in range(num_items):
        prod = df_products.sample(1).iloc[0]
        sku = prod["SKU"]
        cat = prod["Product_Category"]
        seller_id = prod["Seller_ID"]
        supplier_price = prod["Supplier_Price"]
        base_price = prod["Base_Price"]
        discount_pct = prod["Discount_%"]
        rating = prod["Product_Rating"]

        units_sold = random.randint(1, 2)
        effective_price = round(base_price * (1 - discount_pct / 100.0), 2)
        revenue = round(effective_price * units_sold, 2)
        total_order_amt += revenue

        # Detailed Line Items (Flat Sales View compliant)
        order_item_data.append([
            o_id,
            c_id,
            o_date.strftime("%Y-%m-%d %H:%M:%S"),
            state,
            city,
            pincode,
            sku,
            cat,
            seller_id,
            supplier_price,
            base_price,
            discount_pct,
            rating,
            units_sold,
            revenue,
            pay_method,
            status,
        ])

    order_data.append([
        o_id,
        c_id,
        o_date.strftime("%Y-%m-%d %H:%M:%S"),
        state,
        city,
        pincode,
        round(total_order_amt, 2),
        pay_method,
        status,
    ])

df_orders = pd.DataFrame(
    order_data,
    columns=[
        "Order_ID",
        "Customer_ID",
        "Order_Date",
        "Customer_State",
        "City",
        "Pincode",
        "Total_Amount",
        "Payment_Method",
        "Status",
    ],
)

df_order_items = pd.DataFrame(
    order_item_data,
    columns=[
        "Order_ID",
        "Customer_ID",
        "Order_Date",
        "Customer_State",
        "City",
        "Pincode",
        "SKU",
        "Product_Category",
        "Seller_ID",
        "Supplier_Price",
        "Base_Price",
        "Discount_%",
        "Product_Rating",
        "Units_Sold",
        "Revenue",
        "Payment_Method",
        "Status",
    ],
)

# ---------------------------------------------------------
# 5. Product & Feature Analytics Events
# ---------------------------------------------------------
features = [
    "Search_Bar",
    "Category_Filter",
    "Wishlist_Add",
    "WhatsApp_Share",
    "Apply_Coupon",
    "Product_Review_View",
    "Checkout_Click",
]
device_types = ["Android App", "Android App", "Android App", "iOS App", "Web"]

event_data = []
for e_idx in range(1, NUM_EVENTS + 1):
    s_id = f"SESS-{random.randint(100000, 999999)}"
    c_id = f"CUST-{random.randint(1, NUM_CUSTOMERS):05d}"
    feat = random.choice(features)
    dev = random.choice(device_types)
    converted = 1 if feat in ["Checkout_Click", "Apply_Coupon"] else 0
    t_stamp = order_start + timedelta(
        days=random.randint(0, 365),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
    )

    event_data.append([
        s_id,
        c_id,
        feat,
        dev,
        converted,
        t_stamp.strftime("%Y-%m-%d %H:%M:%S"),
    ])

df_events = pd.DataFrame(
    event_data,
    columns=[
        "Session_ID",
        "Customer_ID",
        "Feature_Name",
        "Device_Type",
        "Converted_Flag",
        "Timestamp",
    ],
)

# ---------------------------------------------------------
# 6. Export all datasets to CSV
# ---------------------------------------------------------
df_customers.to_csv("meesho_customers.csv", index=False)
df_products.to_csv("meesho_products.csv", index=False)
df_orders.to_csv("meesho_orders.csv", index=False)
df_order_items.to_csv("meesho_sales_flat_analytics.csv", index=False)
df_events.to_csv("meesho_feature_events.csv", index=False)

print("✅ Big Data CSV Files Created Successfully:")
print(f" 1. meesho_customers.csv          -> {len(df_customers):,} rows")
print(f" 2. meesho_products.csv           -> {len(df_products):,} rows")
print(f" 3. meesho_orders.csv             -> {len(df_orders):,} rows")
print(f" 4. meesho_sales_flat_analytics.csv-> {len(df_order_items):,} rows")
print(f" 5. meesho_feature_events.csv     -> {len(df_events):,} rows")

✅ Big Data CSV Files Created Successfully:
 1. meesho_customers.csv          -> 10,000 rows
 2. meesho_products.csv           -> 500 rows
 3. meesho_orders.csv             -> 50,000 rows
 4. meesho_sales_flat_analytics.csv-> 70,032 rows
 5. meesho_feature_events.csv     -> 150,000 rows
